# Домашнее задание #5 — КардиоРиск AI

Проект для курса **AI Beyond Fit Predict**.

- Датасет: UCI Heart Disease (Cleveland)
- Задача: бинарная классификация риска (`num > 0`)
- Модель: `RandomForestClassifier` + медианная импутация

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from joblib import dump

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay

RANDOM_STATE = 42
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

columns = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"
]

feature_order = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal"
]

feature_labels = {
    "age": "Возраст",
    "sex": "Пол",
    "cp": "Тип боли в груди",
    "trestbps": "Давление в покое",
    "chol": "Холестерин",
    "fbs": "Сахар натощак > 120",
    "restecg": "ЭКГ в покое",
    "thalach": "Макс. пульс",
    "exang": "Стенокардия при нагрузке",
    "oldpeak": "Снижение ST",
    "slope": "Наклон ST",
    "ca": "Число крупных сосудов",
    "thal": "Thalassemia",
}

validation_rules = {
    "age": {"type": "float", "min": 20, "max": 90},
    "sex": {"type": "int", "allowed": [0, 1]},
    "cp": {"type": "int", "allowed": [1, 2, 3, 4]},
    "trestbps": {"type": "float", "min": 80, "max": 250},
    "chol": {"type": "float", "min": 100, "max": 650},
    "fbs": {"type": "int", "allowed": [0, 1]},
    "restecg": {"type": "int", "allowed": [0, 1, 2]},
    "thalach": {"type": "float", "min": 60, "max": 230},
    "exang": {"type": "int", "allowed": [0, 1]},
    "oldpeak": {"type": "float", "min": 0, "max": 8},
    "slope": {"type": "int", "allowed": [1, 2, 3]},
    "ca": {"type": "float", "min": 0, "max": 4},
    "thal": {"type": "int", "allowed": [3, 6, 7]},
}

In [ ]:
raw_df = pd.read_csv(DATA_URL, header=None, names=columns)
raw_df.head()

In [ ]:
df = raw_df.replace("?", np.nan)
for col in columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["num"]).copy()
X = df[feature_order]
y = (df["num"] > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[("num", SimpleImputer(strategy="median"), feature_order)],
    remainder="drop",
    verbose_feature_names_out=False,
)

model = RandomForestClassifier(
    n_estimators=500,
    max_depth=6,
    min_samples_split=8,
    min_samples_leaf=4,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model),
])

pipeline.fit(X_train, y_train)

proba = pipeline.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

accuracy = accuracy_score(y_test, pred)
roc_auc = roc_auc_score(y_test, proba)
report = classification_report(y_test, pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")
print(report)

## Метрики holdout

- Accuracy: **0.8852**
- ROC-AUC: **0.9621**
- Precision (класс 1): **0.8387**
- Recall (класс 1): **0.9286**
- F1 (класс 1): **0.8814**

In [ ]:
RocCurveDisplay.from_predictions(y_test, proba)
plt.title("ROC-кривая на holdout")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
importances = pipeline.named_steps["model"].feature_importances_
feature_importance = {
    f: float(v)
    for f, v in sorted(zip(feature_order, importances), key=lambda kv: kv[1], reverse=True)
}

medians = {f: float(X_train[f].median()) for f in feature_order}
ranges = {
    f: {"min": float(np.nanmin(X_train[f].values)), "max": float(np.nanmax(X_train[f].values))}
    for f in feature_order
}

meta = {
    "model_name": "CardioRisk AI RandomForest",
    "dataset": "UCI Heart Disease (Cleveland)",
    "target_definition": "1 if num > 0 else 0",
    "threshold": 0.5,
    "feature_order": feature_order,
    "feature_labels": feature_labels,
    "validation_rules": validation_rules,
    "medians": medians,
    "ranges": ranges,
    "feature_importance": feature_importance,
    "metrics": {
        "accuracy": float(accuracy),
        "roc_auc": float(roc_auc),
    },
    "train_shape": [int(X_train.shape[0]), int(X_train.shape[1])],
    "test_shape": [int(X_test.shape[0]), int(X_test.shape[1])],
    "random_state": RANDOM_STATE,
}

artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)

dump(pipeline, artifacts_dir / "model.joblib")
with (artifacts_dir / "meta.json").open("w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("Артефакты сохранены в папку artifacts/")
print("Top-5 важных признаков:")
for feat, imp in list(feature_importance.items())[:5]:
    print(f"{feat}: {imp:.4f}")